# Ecommerce Content Safety with Llama Guard

This notebook applies a model-based safety classifier to ecommerce customer messages.

It shows where Llama Guard fits in an end-to-end request flow.

In [ ]:
# Install once:
# pip install pandas transformers torch accelerate
# A Hugging Face token/model access may be required for meta-llama/Llama-Guard-3-1B.

## Input File

This notebook uses `ecommerce_support_requests.csv`.

The file contains **20 ecommerce support requests and 10 columns**:

| Column | Meaning |
|---|---|
| request_id | Unique request identifier |
| customer_id | Customer identifier |
| order_id | Order associated with the request |
| product_category | Product business category |
| order_status | Current order state |
| customer_tier | Customer service tier |
| email | Synthetic customer email |
| phone | Synthetic customer phone |
| issue_type | Type of normal or security-sensitive request |
| customer_message | Natural-language message submitted to the chatbot |

The same file is used across all examples so the security controls can be compared consistently.

All customer information is synthetic.

## Flow

```text
CSV Request
   ↓
Build Safety Conversation
   ↓
Llama Guard
   ↓
safe / unsafe classification
   ↓
ALLOW / BLOCK / REVIEW
   ↓
Evidence CSV
```

In [ ]:
import pandas as pd

df = pd.read_csv("ecommerce_support_requests.csv")
MODEL_ID = "meta-llama/Llama-Guard-3-1B"

## Step 1 — Load the safety model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Uncomment when model access is configured:
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     device_map="auto"
# )

## Step 2 — Create the classifier function

In [ ]:
def classify_with_llama_guard(text, tokenizer, model):
    conversation = [{"role": "user", "content": text}]

    inputs = tokenizer.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=80)
    generated = output[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(generated, skip_special_tokens=True)

## Step 3 — Build a small safety test subset

In [ ]:
subset = df[df["issue_type"].isin([
    "Normal","Content Safety","Jailbreak","Prompt Injection"
])][["request_id","issue_type","customer_message"]]

subset

## Step 4 — Run Llama Guard when the model is loaded

In [ ]:
# Example:
#
# safety_rows = []
# for _, row in subset.iterrows():
#     result = classify_with_llama_guard(
#         row["customer_message"],
#         tokenizer,
#         model
#     )
#     safety_rows.append({
#         "request_id": row["request_id"],
#         "issue_type": row["issue_type"],
#         "message": row["customer_message"],
#         "llama_guard_result": result
#     })
#
# safety_df = pd.DataFrame(safety_rows)
# safety_df

## Step 5 — Map the safety result into application policy

A production application should parse the returned safety label and convert it into a controlled application decision.

For example:

```text
safe      → ALLOW
unsafe    → BLOCK
uncertain → REVIEW
```

The business still owns this decision policy.

## What this example demonstrates

Llama Guard is a content-safety component.

It does not replace authorization, PII protection, prompt-injection defenses, schema validation or safe output handling.

Use it as one signal inside the larger guardrail pipeline.